In [ ]:
# Data 

In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from ScraperFC.sofascore import Sofascore

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [3]:
sf = Sofascore()

def scrape_match(match_id):
    try:
        meta = sf.get_match_dict(match_id)
        df = sf.scrape_player_match_stats(match_id)

        if df is None or df.empty:
            print(f"No data for match_id={match_id}")
            return pd.DataFrame()

        df = df.copy()
        df["match_id"] = match_id
        df["home_team"] = meta.get("homeTeam", {}).get("name")
        df["away_team"] = meta.get("awayTeam", {}).get("name")
        df["home_score"] = meta.get("homeScore", {}).get("current")
        df["away_score"] = meta.get("awayScore", {}).get("current")
        df["tournament"] = meta.get("tournament", {}).get("name")
        df["season"] = meta.get("season", {}).get("name")
        df["status"] = meta.get("status", {}).get("description")
        df["start_time"] = meta.get("startTimestamp")

        return df

    except Exception as e:
        print(f"match_id={match_id} failed: {e}")
        return pd.DataFrame()

In [6]:
# Legacy explicit per-match scraping block (kept as a separate cell)
# England matches
eng_cro_df = scrape_match(15186504)  # England vs Croatia
eng_pan_df = scrape_match(15186676)  # England vs Panama
eng_gha_df = scrape_match(15186672)  # England vs Ghana
eng_con_32_df = scrape_match(12813020)  # England vs DR Congo (R32)
eng_mex_16_df = scrape_match(12813007)  # England vs Mexico (R16)
eng_nor_8_df = scrape_match(12813017)  # England vs Norway (QF)
eng_arg_4_df = scrape_match(12812996)  # England vs Argentina (SF)

# France matches
fra_nor_df = scrape_match(15186537)  # France vs Norway
fra_sen_df = scrape_match(15186501)  # France vs Senegal
fra_ira_df = scrape_match(15186769)  # France vs Iraq
fra_swe_32_df = scrape_match(12812995)  # France vs Sweden (R32)
fra_par_16_df = scrape_match(12813010)  # France vs Paraguay (R16)
fra_mor_8_df = scrape_match(12813016)  # France vs Morocco (QF)
fra_spa_4_df = scrape_match(12813008)  # France vs Spain (SF)

third_place_df = scrape_match(12813003)  # Third-place match

In [7]:
if eng_cro_df.empty:
    raise ValueError("raw_df is empty. Scraping and fallback CSV loading both returned no data.")

print("Rows, Columns:", eng_cro_df.shape)
print("Duplicate column names:", int(eng_cro_df.columns.duplicated().sum()))

null_pct = (eng_cro_df.isna().mean() * 100).sort_values(ascending=False)
display(null_pct.head(15).to_frame("null_pct_top15"))

key_cols = ["name", "teamName", "match_id", "minutesPlayed", "start_time"]
existing_key_cols = [c for c in key_cols if c in eng_cro_df.columns]
display(eng_cro_df[existing_key_cols].head(5))

Rows, Columns: (51, 109)
Duplicate column names: 2


,null_pct_top15
accurateKeeperSweeper,98.039216
goodHighClaim,98.039216
penaltyConceded,98.039216
penaltyFaced,98.039216
errorLeadToAShot,98.039216
totalOffside,98.039216
penaltyWon,98.039216
keeperSaveValue,96.078431
goalkeeperValueNormalized,96.078431
saves,96.078431


,name,teamName,match_id,minutesPlayed,start_time
0,Jordan Pickford,England,15186504,90.0,1781726400
1,Reece James,England,15186504,90.0,1781726400
2,Ezri Konsa,England,15186504,90.0,1781726400
3,John Stones,England,15186504,87.0,1781726400
4,Nico O'Reilly,England,15186504,90.0,1781726400
